# 🔐 CyberSec-FT — Unsloth QLoRA Fine-Tuning

Fine-tunes **Phi-3.5-mini-Instruct** on CVE/Exploit security data using Unsloth.

**Before running:**
1. Set Runtime → Change runtime type → **GPU (T4 or A100)**
2. Upload your `dataset/` folder to Google Drive at:
   `My Drive/CS_FT/dataset/` (train.jsonl, val.jsonl, test.jsonl)
3. Run cells in order

## Cell 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install -q unsloth trl>=0.9.0 transformers>=4.45.0 datasets accelerate peft bitsandbytes pyyaml rouge-score
print('✅ Dependencies installed')

## Cell 2: Check GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
!nvidia-smi

## Cell 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Verify dataset is in Drive
DATASET_DIR = '/content/drive/MyDrive/CS_FT/dataset'
for f in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    path = os.path.join(DATASET_DIR, f)
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    status = f'{size_mb:.1f} MB' if size_mb else '❌ NOT FOUND'
    print(f'  {f}: {status}')

## Cell 4: Clone Repo

In [ ]:
import os

REPO_URL = 'https://github.com/Mohamedabul/CS_FT.git'
REPO_DIR = '/content/CS_FT'

if os.path.exists(REPO_DIR):
    print('Repo already exists — pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## Cell 5: Link Dataset from Drive

In [ ]:
import os, shutil

DATASET_SRC = '/content/drive/MyDrive/CS_FT/dataset'
DATASET_DST = '/content/CS_FT/dataset'

os.makedirs(DATASET_DST, exist_ok=True)

# Symlink each file (faster than copying)
for f in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    src = os.path.join(DATASET_SRC, f)
    dst = os.path.join(DATASET_DST, f)
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f'  ✅ {f} linked')

## Cell 6: Configure Training Paths

In [ ]:
import yaml

# Update training_config.yaml Colab paths
config_path = '/content/CS_FT/configs/training_config.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg['colab']['output_dir']   = '/content/drive/MyDrive/CS_FT/models/adapter'
cfg['colab']['dataset_path'] = '/content/CS_FT/dataset'

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('✅ Config updated with Colab paths')
print(f"  Output dir : {cfg['colab']['output_dir']}")
print(f"  Dataset    : {cfg['colab']['dataset_path']}")

## Cell 7: Run Training 🚀

Checkpoints are saved to Google Drive every 200 steps.  
If training stops, just re-run this cell — it auto-resumes.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '/content/CS_FT/training/unsloth/train_unsloth.py'],
    cwd='/content/CS_FT/training/unsloth'
)
print('\n✅ Training complete!' if result.returncode == 0 else f'\n❌ Error (code {result.returncode})')

## Cell 8: Quick Inference Test

In [ ]:
import sys
sys.path.insert(0, '/content/CS_FT/training/unsloth')

from model_loader_unsloth import load_for_inference

ADAPTER_DIR = '/content/drive/MyDrive/CS_FT/models/adapter'
model, tokenizer = load_for_inference(ADAPTER_DIR, max_seq_length=512)

# Test prompt
instruction = 'Analyze the following CVE and provide a structured vulnerability report.'
context = (
    'CVE ID: CVE-2021-44228\n'
    'Description: Apache Log4j2 JNDI features allow remote code execution.\n'
    'CVSS v3: 10.0 CRITICAL | Attack Vector: NETWORK | CWE: CWE-917'
)

messages = [{'role': 'user', 'content': f'{instruction}\n\n{context}'}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

import torch
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=400, temperature=0.7, do_sample=True)

generated = output[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(generated, skip_special_tokens=True))

## Cell 9: Verify Saved Files

In [ ]:
import os

for folder in [
    '/content/drive/MyDrive/CS_FT/models/adapter',
    '/content/drive/MyDrive/CS_FT/models/merged',
]:
    if os.path.exists(folder):
        files = os.listdir(folder)
        size = sum(os.path.getsize(os.path.join(folder, f)) for f in files) / 1e6
        print(f'✅ {folder}\n   {len(files)} files | {size:.0f} MB')
    else:
        print(f'❌ {folder} not found')